# Pipelines de procesamiento para el Dataset de Reseñas de productos y cuidado de la piel de Sephora

Este notebook construye un pipeline completo de limpieza y transformación para el dataset de `data/raw/`.

### Transformadores Personalizados:
Para solucionar los problemas de calidad de datos detectados en la fase del EDA, hemos construido una arquitectura de transformadores personalizados. Esto asegura que la limpieza, replicable y evite la fuga de información.

Los transformadores definidos para el modelo de Sephora son:

1. **`DropColumnsTransformer`**: Elimina variables que determinamos como inútiles desde el inicio (ej. el índice residual `Unnamed: 0`).
2. **`ColumnRenamerTransformer`**: Estandariza los nombres de las columnas que quedaron duplicadas con sufijos `_x` e `_y` tras la integración de los datasets de productos y reseñas.
3. **`UnknownToNaNTransformer`**: Estandariza strings como `'unknown'`, `'UNKNOWN'` o espacios en blanco, transformándolos en verdaderos valores nulos (`np.nan`).
4. **`DropHighMissingTransformer`**: Identifica y elimina automáticamente las columnas que superen el 90% de nulos (umbral definido tras detectar que variables como `variation_desc` estaban casi vacías).
5. **`OutlierCapper`**: Aplica la técnica de *Capping* (IQR) a las variables continuas para limitar valores extremos, excluyendo de forma inteligente las variables binarias y los IDs para no deformar su naturaleza.
6. **`DropZeroVarianceTransformer`**: Descarta variables numéricas que tengan varianza cero (el mismo valor en todas las filas), ya que no aportan poder predictivo a los modelos.
7. **`SmartImputerTransformer`**: Imputa valores faltantes según el tipo de dato (mediana para numéricas, moda para categóricas). Además, incorpora una regla: los nulos en la métrica `helpfulness` se rellenan con `0`, asumiendo que representan reseñas sin votos.
8. **`PriceCleanerTransformer`**: Transformador específico para limpiar las variables de precio (ej. `price_usd`), eliminando símbolos de moneda (`$`) o comas, y convirtiéndolas a formato numérico (`float`).
9. **`TextCleanerTransformer`**: Elimina saltos de línea (`\n`) y espacios residuales en las variables categóricas de texto utilizando `.strip()`.

In [ ]:
# Se configura para que detecte y se ejecute Google Colab o local y definir los transformadores localmente
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn import set_config
from sklearn.base import BaseEstimator, TransformerMixin

# Se ve la lógica para que sea compatible entre Colab y el entorno local
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print('Google Colab detectado. Sube tu dataset unificado de Sephora cuando se te pida.')
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        data_filename = list(uploaded.keys())[0]
    else:
        raise FileNotFoundError('Debe subir el dataset en Colab antes de ejecutar el notebook.')
else:
    # Ajustado a la posible ruta de tu dataset unificado
    data_filename = '../data/raw/' 

print('Usando ruta de datos:', data_filename)

# Se definen los transformadores personalizados

class DropColumnsTransformer(BaseEstimator, TransformerMixin):
    """Elimina columnas específicas del DataFrame que no aportan valor."""
    def __init__(self, columns_to_drop):
        self.columns_to_drop = columns_to_drop

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_copy = X.copy()
        # Se filtran solo las columnas que realmente existen para evitar errores
        cols = [col for col in self.columns_to_drop if col in X_copy.columns]
        return X_copy.drop(columns=cols)


class ColumnRenamerTransformer(BaseEstimator, TransformerMixin):
    """Renombra columnas cruzadas tras un merge (ej. price_usd_x -> price_usd)."""
    def __init__(self, rename_dict):
        self.rename_dict = rename_dict

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_copy = X.copy()
        return X_copy.rename(columns=self.rename_dict)


class UnknownToNaNTransformer(BaseEstimator, TransformerMixin):
    """Convierte el string 'unknown' o vacíos a un valor nulo real (np.nan)."""
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.replace(['unknown', 'UNKNOWN', 'Unknown', ''], np.nan)


class DropHighMissingTransformer(BaseEstimator, TransformerMixin):
    """Elimina las columnas que superen un umbral máximo de valores nulos."""
    def __init__(self, threshold=0.90): # Subimos el umbral al 90%
        self.threshold = threshold
        self.cols_to_drop_ = []

    def fit(self, X, y=None):
        pct_nulos = X.isnull().mean()
        self.cols_to_drop_ = pct_nulos[pct_nulos > self.threshold].index.tolist()
        return self

    def transform(self, X):
        X_copy = X.copy()
        cols = [c for c in self.cols_to_drop_ if c in X_copy.columns]
        return X_copy.drop(columns=cols)


class OutlierCapper(BaseEstimator, TransformerMixin):
    """Aplica capping a los valores atípicos usando el método IQR."""
    def __init__(self, apply_capping=True):
        self.apply_capping = apply_capping
        self.bounds_ = {}

    def fit(self, X, y=None):
        if not self.apply_capping:
            return self
        
        # Excluimos IDs y binarias para no deformarlas
        columnas_a_excluir = ['author_id', 'product_id', 'brand_id', 'is_recommended', 
                              'limited_edition', 'new', 'online_only', 'out_of_stock', 'sephora_exclusive']
        
        num_cols = [col for col in X.select_dtypes(include=['number']).columns if col not in columnas_a_excluir]
        
        for col in num_cols: 
            Q1 = X[col].dropna().quantile(0.25)
            Q3 = X[col].dropna().quantile(0.75)
            IQR = Q3 - Q1
            self.bounds_[col] = (Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)
        return self

    def transform(self, X):
        X_copy = X.copy()
        if not self.apply_capping:
            return X_copy
        for col, (lower, upper) in self.bounds_.items():
            if col in X_copy.columns:
                X_copy[col] = np.clip(X_copy[col], lower, upper)
        return X_copy


class DropZeroVarianceTransformer(BaseEstimator, TransformerMixin):
    """Elimina columnas numéricas con varianza igual a 0 (un solo valor constante)."""
    def __init__(self):
        self.cols_to_drop_ = []

    def fit(self, X, y=None):
        num_cols = X.select_dtypes(include=['number']).columns
        zero_variance = [col for col in num_cols if X[col].std() == 0]
        self.cols_to_drop_ = zero_variance
        return self

    def transform(self, X):
        X_copy = X.copy()
        cols = [c for c in self.cols_to_drop_ if c in X_copy.columns]
        return X_copy.drop(columns=cols)


class SmartImputerTransformer(BaseEstimator, TransformerMixin):
    """Imputa nulos: Mediana para numéricas, Moda para categóricas."""
    def __init__(self, low_threshold=0.10):
        self.low_threshold = low_threshold
        self.cols_simples_ = []
        self.cols_complejas_ = []

    def fit(self, X, y=None):
        porcentaje_nulos = X.isnull().mean()
        for col in X.columns:
            pct = porcentaje_nulos[col]
            if 0 < pct <= self.low_threshold:
                self.cols_simples_.append(col)
            elif pct > self.low_threshold:
                self.cols_complejas_.append(col)
        print(f"SmartImputer - Simples (<10%): {self.cols_simples_}")
        print(f"SmartImputer - Complejas (>10%): {self.cols_complejas_}")
        return self

    def transform(self, X):
        X_copy = X.copy()
        todas_las_cols = self.cols_simples_ + self.cols_complejas_
        
        for col in todas_las_cols:
            if col in X_copy.columns:
                if col == 'helpfulness':
                    X_copy[col] = X_copy[col].fillna(0)
                elif pd.api.types.is_numeric_dtype(X_copy[col]):
                    X_copy[col] = X_copy[col].fillna(X_copy[col].median())
                else:
                    X_copy[col] = X_copy[col].fillna(X_copy[col].mode()[0])
        return X_copy


class PriceCleanerTransformer(BaseEstimator, TransformerMixin):
    """Limpia símbolos de moneda y convierte a formato numérico (float)."""
    def __init__(self, column='price_usd'):
        self.column = column

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_copy = X.copy()
        if self.column in X_copy.columns:
            # Primero nos aseguramos de que sea string
            X_copy[self.column] = X_copy[self.column].astype(str)
            X_copy[self.column] = X_copy[self.column].replace('[$,]', '', regex=True)
            X_copy[self.column] = pd.to_numeric(X_copy[self.column], errors='coerce')
        return X_copy


class TextCleanerTransformer(BaseEstimator, TransformerMixin):
    """Elimina espacios en blanco residuales y saltos de línea de variables tipo texto."""
    def __init__(self, columns=None):
        self.columns = columns

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_copy = X.copy()
        cols_to_clean = self.columns if self.columns else X_copy.select_dtypes(include=['object']).columns
        for col in cols_to_clean:
            if col in X_copy.columns:
                X_copy[col] = X_copy[col].astype(str).str.strip()
        return X_copy

print('Entorno y transformadores definidos localmente para el dataset de Sephora.')

### Observaciones de la Carga de Datos Crudos

Al visualizar las primeras filas del dataset unificado, podemos confirmar que la estrategia de ingesta fue exitosa:

1. **Captura Efectiva de Nulos:** Debido al parámetro `na_values`, los espacios vacíos y textos residuales (como `'unknown'` o `'null'`) fueron transformados automáticamente en valores `NaN` reales desde la carga (como se observa en las columnas `helpfulness` y `skin_tone`).
2. **Estado "Salvaje" Intencional:** Mantenemos las columnas residuales (como `Unnamed: 0`) y problemas de formato en los precios. No aplicaremos ninguna limpieza manual sobre este DataFrame.

**El objetivo de un Pipeline fuerte es la automatización total.** Al cargar los datos en su estado crudo, simulamos un entorno de producción real donde el sistema debe ser capaz de ingerir datos nuevos y procesarlos de principio a fin sin intervención humana.

In [ ]:
# Cargamos los datos que estan juntos desde el archivo seleccionado o la ruta local
ruta = data_filename

# Leemos el CSV, interpretando los espacios vacíos o textos 'null' como verdaderos nulos
# Agregamos también un espacio en blanco ' ' a la lista en caso de
df_raw = pd.read_csv(ruta, na_values=['NULL', 'null', '', ' ', 'unknown', 'UNKNOWN'])

# Nosotros  ya no convertimos los precios a numéricos ni hacemos limpieza manual previa.
# Dejaremos que los transformadores personalizados se encarguen

print('Dimensiones de los datos crudos:', df_raw.shape)
print('\nMuestra de las primeras 5 filas:')
display(df_raw.head())

## Asignación de los datos

### Definición de Rutas

Para que nuestro Pipeline funcione de manera segura y sin errores, debemos enseñarle a distinguir entre los diferentes tipos de datos presentes en el dataset de Sephora. Los tratamientos que recibirán son distintos:

* **Ruta Numérica (`variables_numericas`):** Incluye el precio y las métricas de interacción (`loves_count`, `reviews`, etc.). Estas columnas pasarán por transformaciones estadísticas: cálculo de medianas para imputar nulos, cálculo de cuartiles para el *Capping* de outliers y limpieza de símbolos de moneda.
* **Ruta Categórica (`variables_categoricas`):** Incluye descriptores de texto como `brand_name_x` o `skin_type`. Estas columnas pasarán por limpieza de caracteres (espacios, saltos de línea) y una imputación de nulos basada en la moda (la categoría más frecuente).

Estas listas alimentarán a nuestro `ColumnTransformer`, el organizador principal que se encargará de dirigir cada columna por su ruta correspondiente en paralelo.

In [ ]:
# Definimos explícitamente qué columnas van por cual ruta en el Pipeline
# Tenemos que usar los nombres reales del dataset

variables_numericas = [
    'price_usd',
    'rating',
    'helpfulness',
    'total_feedback_count',
    'total_neg_feedback_count',
    'total_pos_feedback_count'
]

variables_categoricas = [
    'brand_name',
    'skin_type',
    'skin_tone',
    'eye_color',
    'hair_color'
]

# Se imprimen los resultados para verificar que las variables se asignaron correctamente
print('Variables numéricas asignadas:', variables_numericas)
print('Variables categóricas asignadas:', variables_categoricas)

## Diagrama interactivo

### Ejecución de los datos crudos a los datos listos

Al utilizar el método `.fit_transform()` sobre nuestro DataFrame crudo (`df_raw`), el notebook ejecuta automáticamente todas las operaciones en el orden exacto que diseñamos:

1. **Limpieza Estructural y Formato:** Elimina columnas basura, ajusta textos y corrige los formatos numéricos de los precios extrayendo solo el valor real.
2. **Gestión Inteligente de Nulos:** Aplica nuestras reglas (medias, modas y constantes) para imputar los valores faltantes sin perder valiosas reseñas.
3. **Preprocesamiento Matemático (Bifurcación):** El `ColumnTransformer` separa los caminos. Los números se ajustan con *Capping* (para contener outliers), se revisa su varianza y se estandarizan; mientras tanto, las categorías de texto se transforman en matrices binarias.

In [ ]:
# 1. Ruta para números
num_pipe = Pipeline([
    ('capper', OutlierCapper(apply_capping=True)),
    ('zero_variance', DropZeroVarianceTransformer()),
    ('scaler', StandardScaler())
])

# 2. Ruta para textos
cat_pipe = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 3. El enrutador
preprocesador = ColumnTransformer(
    transformers=[
        ('num', num_pipe, variables_numericas),
        ('cat', cat_pipe, variables_categoricas)
    ], remainder='drop' # Esto descarta automáticamente todo lo que no esté en nuestras listas
)

# 4. Pipeline de Sephora
pipeline_ml_completo = Pipeline([
    
    ('drop_basura', DropColumnsTransformer(columns_to_drop=[
        'Unnamed: 0', 'author_id', 'product_id', 'brand_id', 
        'product_name_y', 'brand_name_y', 'price_usd_y', 'rating_y'
    ])),

    ('renombrar', ColumnRenamerTransformer({
        'price_usd_x': 'price_usd',
        'brand_name_x': 'brand_name'
    })),

    ('limpieza_unknowns', UnknownToNaNTransformer()),
    ('limpia_textos', TextCleanerTransformer(columns=variables_categoricas)),
    ('limpia_precios', PriceCleanerTransformer(column='price_usd')),

    ('drop_high_nan', DropHighMissingTransformer(threshold=0.90)),
    ('smart_imputer', SmartImputerTransformer(low_threshold=0.10)),

    ('preprocesamiento', preprocesador)
])

# Y finalmente se muestra el diagrama interactivo
set_config(display='diagram')
pipeline_ml_completo

In [ ]:
print(df_raw.columns.tolist())

### Transformación Exitosa y Verificación Final

Al ejecutar nuestro Pipeline sobre el dataset, los registros de salida y la matriz confirman que nuestra arquitectura procesó los datos perfectamente:

1. **Imputación Dinámica (`SmartImputer`):** Los logs del sistema demuestran que el transformador analizó en tiempo real el porcentaje de nulos. Clasificó automáticamente las columnas en rutas "Simples" (< 10% de nulos) y "Complejas" (> 10%).
2. **Escalado de Variables Numéricas (`StandardScaler`):** Al observar las primeras columnas de la tabla final (como `price_usd` o `rating`), notamos que los valores originales fueron reemplazados por decimales positivos y negativos. Esto indica que las variables continuas fueron estandarizadas correctamente (centradas en cero con desviación estándar de 1).
3. **Expansión Binarizada (`One-Hot Encoding`):** El dataset pasó de tener un puñado de variables iniciales a un total de **148 columnas**. Este salto dimensional es el resultado de codificar variables categóricas de texto (como marcas, tonos de piel o colores de ojos) en múltiples columnas binarias independientes (`0.0` y `1.0`).
4. **Limpieza de Nomenclatura:** Se aplicó una limpieza final a nivel de código para remover los prefijos técnicos. De esta forma, el dataset final preserva nombres de variables legibles y directos (ej. `brand_name_CLINIQUE`), facilitando la interpretabilidad de los modelos a futuro.

In [ ]:
X_transformado = pipeline_ml_completo.fit_transform(X, y)
def fit(self, X, y=None):
    self.cols_simples_ = []
    self.cols_complejas_ = []

preprocesador = pipeline_ml_completo.named_steps['preprocesamiento']

# numéricas (no cambian nombre)
nombres_num = variables_numericas

# categóricas (OneHotEncoder)
encoder = preprocesador.named_transformers_['cat']['onehot']
nombres_cat = encoder.get_feature_names_out(variables_categoricas)

# unir todo
nombres_finales = list(nombres_num) + list(nombres_cat)

df_limpio = pd.DataFrame(X_transformado, columns=nombres_finales)

### Limpieza y Verificación de la Matriz Final

Como último paso de la transformación, aplicamos una limpieza a los nombres de las variables. Por defecto, el `ColumnTransformer` inyecta prefijos (como `num__` y `cat__`) para rastrear la ruta por la que pasó cada columna en el Pipeline. Al removerlos mediante comprensión de listas, recuperamos nombres limpios, legibles y listos para producción (por ejemplo, `brand_name_CLINIQUE` en lugar de `cat__brand_name_CLINIQUE`).

Al observar la tabla final, podemos confirmar visualmente el éxito de nuestra arquitectura de preprocesamiento:

* **Variables Numéricas Estandarizadas:** Columnas como `price_usd` y `rating` ya no muestran sus valores originales, sino valores decimales positivos y negativos. Esto confirma que el `StandardScaler` actuó correctamente, ajustando las distribuciones para que tengan media cero y desviación estándar de uno.
* **Expansión Binarizada (One-Hot Encoding):** El dataset pasó de muchas de variables a un total de **148 columnas**. Características de texto vitales para el negocio (como marcas, tonos de piel o colores de cabello) han sido separadas en columnas independientes compuestas exclusivamente por `0.0` y `1.0`.

In [ ]:
# 1. Limpiamos los prefijos 'num__' y 'cat__' de los nombres de las columnas
nombres_limpios = [nombre.replace('num__', '').replace('cat__', '') for nombre in df_limpio.columns]

# 2. Le asignamos estos nombres limpios y bonitos a nuestro DataFrame final
df_limpio.columns = nombres_limpios

# 3. Imprimimos el resultado (mostrando solo una muestra para no saturar la pantalla)
print(f'Total de variables finales listas para el modelo: {len(df_limpio.columns)}\n')

print("Muestra de las columnas resultantes:")
# Imprimimos las primeras 20 para ver cómo quedaron (los números y las primeras categorías)
for i, nombre in enumerate(df_limpio.columns[:20]): 
    print(f'{i+1:02d}. {nombre}')

if len(df_limpio.columns) > 20:
    print(f"\n... y {len(df_limpio.columns) - 20} variables categóricas más generadas por el One-Hot Encoder.")

# Vemos la tabla final perfecta
display(df_limpio.head())

### Resumen del Súper Pipeline de Sephora

El preprocesamiento de nuestros datos se ejecutó a través de un Pipeline que asegura reproducibilidad y evita la fuga de datos (Data Leakage). El flujo consta de los siguientes pasos:

- **Paso A (Limpieza Estructural):** Se eliminan columnas residuales producto del cruce de tablas (las terminadas en `_y`), identificadores únicos (IDs) y metadatos que no aportan valor predictivo para el modelo.
- **Paso B (Limpieza de Formatos):** Se estandarizan las cadenas de texto y se procesan los precios numéricos (`PriceCleanerTransformer`)-
- **Paso C (Gestión Avanzada de Nulos):** Se convierten valores de texto como "unknown" en nulos reales (`NaN`). Se descartan columnas irrecuperables (más del 90% de nulos) y se aplica un **Imputador Inteligente** que evalúa dinámicamente si usar imputación simple o compleja dependiendo de si los nulos superan el umbral del 10%.
- **Paso D (Enrutamiento Matemático - Numéricas):** A variables como el precio o el conteo de reseñas se les aplica *Capping* para contener valores atípicos (outliers), se filtran las de varianza cero, y se escalan con `StandardScaler` para equilibrar sus magnitudes.
- **Paso E (Enrutamiento Matemático - Categóricas):** Las variables de perfil del usuario y del producto (como `brand_name`, `primary_category`, `skin_type`, `skin_tone`, etc.) se expanden de forma binaria mediante `OneHotEncoder`.